# 3. Label and data statistics

*Authors: La Anh Hào, Võ Nguyễn Vân Anh, Lưu Minh Quân · data: `m3_data_summary.json`, `m3_labels.json`,
`m3_selection.csv`, `m3_split.json`, `m3_cleaning_report.csv`, `m3_feature_report.csv`*

This chapter analyses the Milestone 3 dataset before modelling: how many papers are eligible (3.1), what the
20 manually labelled papers look like (3.2), how the automatic matcher was tuned and validated (3.3), and how
cleaning completeness and each of the 25 engineered features separate positive from negative candidate pairs
(3.4). Every table, number and interactive Plotly chart is computed from `data/m3_*` at build time.

In [ ]:
import ast
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import report_data as rd

rd.check()
ds = rd.data_summary()
lbl = rd.labels()
sel = rd.selection()
sp = rd.split()
clean = rd.cleaning_report()
feat = rd.feature_report()
req = rd.requirements()
s = ds["summary"]
rd.md(
    f"Input snapshot `data/m3/` version `{ds['data_version']}` (signature `{ds['signature']}`, built "
    f"{ds['built_at'][:16].replace('T', ' ')} UTC in {rd.hm(ds['seconds'])}) locked to parser "
    f"`{ds['parser_version']}` across **{ds['papers']:,}** papers.")

## 3.1 Corpus eligibility and sampling pools

A publication is eligible for Milestone 3 when it has at least one Milestone 2 BibTeX entry and at least one
arXiv-keyed Semantic Scholar reference (`n_entries >= 1` and `n_refs >= 1`).

In [ ]:
rows = [
    ("Total publications", f"{s['papers']:,}"),
    ("Papers with M2 parsed text (`ok` / `empty_source` / `no_root`)",
     f"{s['m2_status']['ok']:,} / {s['m2_status']['empty_source']:,} / {s['m2_status']['no_root']:,}"),
    ("Papers with S2 status (`ok` / `not_found`)",
     f"{s['s2_status']['ok']:,} / {s['s2_status']['not_found']:,}"),
    ("Papers with ≥ 1 BibTeX entry (`.bib` / `\\bibitem`)",
     f"{s['papers_with_entries']:,} ({s['bib_source_of_papers_with_entries']['bib']:,} / "
     f"{s['bib_source_of_papers_with_entries']['bibitem']:,})"),
    ("Papers with ≥ 1 arXiv reference", f"{s['papers_with_arxiv_refs']:,}"),
    ("Eligible papers (≥ 1 entry and ≥ 1 arXiv reference)", f"{s['eligible']:,} ({100 * s['eligible_share']:.2f}%)"),
    ("Total BibTeX entries (`entries.parquet`) / in eligible papers",
     f"{s['entries']:,} / {s['entries_in_eligible']:,}"),
    ("Total S2 references / arXiv-keyed (`refs.parquet`)",
     f"{s['refs_all']:,} / {s['arxiv_refs']:,} ({100 * s['arxiv_ref_share']:.2f}%)"),
    ("Median entries / median arXiv references per eligible paper",
     f"{s['median_entries_per_eligible']:.0f} / {s['median_arxiv_refs_per_eligible']:.0f}"),
    ("Total candidate pairs across all eligible papers", f"{s['candidate_pairs_in_eligible']:,}"),
    ("`MANUAL_POOL` (20–60 entries, ≥ 30 arXiv refs) / `SPLIT_POOL`",
     f"{s['manual_pool']:,} / {s['split_pool']:,}"),
]
rd.table(pd.DataFrame(rows, columns=["Quantity", "Value"]), "Table 3.1. Milestone 3 input corpus summary (`m3_data_summary.json`).")

mem_rows = []
for m, d in s["by_member"].items():
    mem_rows.append({
        "Member": m,
        "ID range": rd.MEMBERS[m][1],
        "Papers": d["papers"],
        "Eligible": d["eligible"],
        "Eligible share": 100 * d["eligible"] / d["papers"],
        "Entries": d["entries"],
        "arXiv refs": d["arxiv_refs"],
        "Manual pool": d["manual_pool"],
        "… .bib": d["manual_pool_bib"],
        "… \\bibitem": d["manual_pool_bibitem"],
    })
t_mem = pd.DataFrame(mem_rows)
rd.table(t_mem, "Table 3.2. Eligibility, entries, arXiv references and manual pool by member.",
         {"Eligible share": "{:.1f}%"})

fig = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.14,
    subplot_titles=("Eligible vs ineligible papers", "Manual pool by bibliography source"),
)
fig.add_trace(go.Bar(
    x=t_mem["Member"], y=t_mem["Eligible"], name="Eligible", marker_color=rd.BLUE,
    hovertemplate="%{x} · Eligible<br><b>%{y:,}</b> papers<extra></extra>",
), row=1, col=1)
fig.add_trace(go.Bar(
    x=t_mem["Member"], y=t_mem["Papers"] - t_mem["Eligible"], name="Ineligible", marker_color=rd.AXIS,
    hovertemplate="%{x} · Ineligible<br><b>%{y:,}</b> papers<extra></extra>",
), row=1, col=1)
fig.add_trace(go.Bar(
    x=t_mem["Member"], y=t_mem["… .bib"], name="Manual pool (.bib)", marker_color=rd.AQUA,
    hovertemplate="%{x} · .bib pool<br><b>%{y:,}</b> papers<extra></extra>",
), row=1, col=2)
fig.add_trace(go.Bar(
    x=t_mem["Member"], y=t_mem["… \\bibitem"], name="Manual pool (\\bibitem)", marker_color=rd.ORANGE,
    hovertemplate="%{x} · \\bibitem pool<br><b>%{y:,}</b> papers<extra></extra>",
), row=1, col=2)
fig.update_layout(
    barmode="stack",
    title="Milestone 3 corpus eligibility and manual pool across members",
    legend=dict(orientation="h", yanchor="top", y=-0.22, xanchor="center", x=0.5),
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(title_text="Papers", row=1, col=1)
fig.update_yaxes(title_text="Papers", row=1, col=2)
rd.show(fig, height=360, margin=dict(l=55, r=25, t=65, b=80))
rd.figure_caption("Figure 3.1. Left: eligible vs ineligible papers per member (5,000 IDs each). "
                  "Right: composition of `MANUAL_POOL` by bibliography source.")

## 3.2 Manual ground truth (20 publications)

Each member manually labelled 5 publications from their own range (20 in total). Every entry of each paper
was inspected; a matched `(entry, reference)` pair counts towards the ≥ 20/paper and ≥ 400 total quotas.

In [ ]:
part_of = {}
for p_name in ("test", "valid"):
    for aid in sp[p_name]["manual"]:
        part_of[aid] = p_name

mp = pd.DataFrame(lbl["manual_papers"]).merge(sel[["arxiv_id", "bib_source"]], on="arxiv_id", how="left")
mp["partition"] = mp["arxiv_id"].map(lambda x: part_of.get(x, "train"))
mp["unmatched"] = mp["entries"] - mp["matched"]
mp["match_rate"] = 100 * mp["matched"] / mp["entries"]
mp = mp.sort_values(["member", "arxiv_id"]).reset_index(drop=True)

t_mp = mp[["member", "arxiv_id", "bib_source", "partition", "entries", "n_refs", "matched", "unmatched", "match_rate", "unsure"]].copy()
t_mp.columns = ["Member", "Paper ID", "Bib source", "Partition", "Entries", "arXiv refs", "Matched pairs",
                "No match", "Match rate", "Unsure"]
rd.table(t_mp, "Table 3.3. The 20 manually labelled publications (`m3_labels.json` + `m3_selection.csv`).",
         {"Match rate": "{:.1f}%"})

m_tot = lbl["manual"]
rd.md(
    f"Across all **{m_tot['papers']} manual papers** (5 per member; "
    f"{int((mp['bib_source'] == 'bib').sum())} `.bib` and {int((mp['bib_source'] == 'bibitem').sum())} `\bibitem`), "
    f"we labelled **{m_tot['entries']:,} entries** and confirmed **{m_tot['matched']:,} matched pairs** "
    f"(minimum **{int(mp['matched'].min())}** pairs in any paper, maximum **{int(mp['matched'].max())}**). "
    f"The no-match share is **{100 * m_tot['no_match_share']:.2f}%** ({m_tot['entries'] - m_tot['matched']} entries "
    f"citing works absent from Semantic Scholar's arXiv list), and only **{int(mp['unsure'].sum())} entry** "
    f"({100 * m_tot['unsure_share']:.2f}%, `2605.08524`) was flagged `unsure`.")

subsets = [("All 20 papers", mp)] + [(f"{m} (5 papers)", mp[mp["member"] == m]) for m in rd.MEMBERS]
fig = go.Figure()
buttons = []
for idx, (lbl_s, sub) in enumerate(subsets):
    x_lbl = [f"{r.arxiv_id} ({r.member})" for r in sub.itertuples()]
    fig.add_trace(go.Bar(
        x=x_lbl, y=sub["matched"].tolist(), name="Matched pairs", marker_color=rd.BLUE,
        visible=(idx == 0), showlegend=(idx == 0),
        customdata=np.stack([sub["bib_source"], sub["partition"], sub["match_rate"]], axis=-1),
        hovertemplate="<b>%{x}</b> (%{customdata[0]}, %{customdata[1]})<br>Matched: <b>%{y}</b> pairs "
                      "(%{customdata[2]:.1f}%)<extra></extra>",
    ))
    fig.add_trace(go.Bar(
        x=x_lbl, y=sub["unmatched"].tolist(), name="No match (NIL)", marker_color=rd.AXIS,
        visible=(idx == 0), showlegend=(idx == 0),
        hovertemplate="<b>%{x}</b><br>Unmatched: <b>%{y}</b> entries<extra></extra>",
    ))
    vis = [False] * (2 * len(subsets))
    vis[2 * idx] = True
    vis[2 * idx + 1] = True
    buttons.append(dict(label=lbl_s, method="update", args=[{"visible": vis}]))

fig.add_hline(
    y=20, line_dash="dash", line_color=rd.ORANGE, line_width=1.6,
    annotation_text="quota: ≥ 20 matched pairs/paper", annotation_position="top right",
)
fig.update_layout(
    barmode="stack",
    title="Matched pairs vs unmatched entries per manual publication",
    updatemenus=[dict(active=0, buttons=buttons, x=1.0, xanchor="right", y=1.18, yanchor="top")],
    legend=dict(orientation="h", yanchor="top", y=-0.30, xanchor="center", x=0.5),
)
fig.update_xaxes(tickangle=-35, showgrid=False)
fig.update_yaxes(title_text="BibTeX entries")
rd.show(fig, height=400, margin=dict(l=55, r=25, t=75, b=110))
rd.figure_caption("Figure 3.2. Matched pairs (blue) and unmatched entries (grey) across the 20 manually labelled "
                  "publications; every paper exceeds the 20-pair floor (orange line). Use the dropdown to filter by member.")

## 3.3 Automatic matcher tuning, validation and split

The deterministic rule matcher (`21_labeling`) was tuned strictly on the 12 train manual papers and evaluated
against all 20 manual papers and against the 8 held-out (test + valid) manual papers that tuning never saw.

In [ ]:
tune = pd.DataFrame(lbl["tuning"])
chosen_p = lbl["params"]
fig = go.Figure()
for mt, color in zip(sorted(tune["min_tokens"].unique()), [rd.BLUE, rd.AQUA, rd.ORANGE]):
    sub = tune[tune["min_tokens"] == mt].sort_values("fuzzy")
    fig.add_trace(go.Scatter(
        x=sub["fuzzy"], y=100 * sub["recall"], mode="lines+markers",
        name=f"min_tokens = {mt}", line=dict(color=color, width=2), marker=dict(size=8),
        customdata=np.stack([100 * sub["precision"], 100 * sub["f1"]], axis=-1),
        hovertemplate=f"fuzzy = %{{x}}, min_tokens = {mt}<br>Precision: %{{customdata[0]:.2f}}%<br>"
                      "Recall: <b>%{y:.2f}%</b><br>F1: %{customdata[1]:.2f}%<extra></extra>",
    ))
fig.add_annotation(
    x=chosen_p["fuzzy"], y=100 * tune.loc[(tune["fuzzy"] == chosen_p["fuzzy"]) & (tune["min_tokens"] == chosen_p["min_tokens"]), "recall"].iloc[0],
    text=f"chosen ({chosen_p['fuzzy']}, {chosen_p['min_tokens']})", showarrow=True, arrowhead=2, ay=-30, ax=40,
    font=dict(color=rd.INK),
)
fig.update_layout(
    title="Automatic matcher tuning on 12 train manual papers (all precision = 100%)",
    legend=dict(orientation="h", yanchor="top", y=-0.22, xanchor="center", x=0.5),
)
fig.update_xaxes(title_text="Fuzzy title threshold (token_sort_ratio)", tickvals=[80, 85, 90, 95])
fig.update_yaxes(title_text="Train recall (%)", range=[98.0, 100.2])
rd.show(fig, height=340, margin=dict(l=55, r=25, t=55, b=75))
rd.figure_caption("Figure 3.3. Grid search of `fuzzy` and `min_tokens` on the 12 train manual papers; every "
                  "configuration achieves 100% precision, so `(fuzzy=80, min_tokens=3, year_tol=1)` maximizes recall.")

avm = lbl["auto_vs_manual"]
ho = lbl["auto_vs_manual_held_out"]
t_val = pd.DataFrame([
    ("All manual papers", avm["papers"], avm["entries"], avm["precision"], avm["recall"], avm["f1"], avm["entry_accuracy"]),
    ("Held-out manual (test + valid)", ho["papers"], ho["entries"], ho["precision"], ho["recall"], ho["f1"], ho["entry_accuracy"]),
], columns=["Evaluation set", "Papers", "Entries", "Precision", "Recall", "F1", "Entry accuracy"])
rd.table(t_val, "Table 3.4. Automatic rule matcher evaluated against manual ground truth (`m3_labels.json`).",
         {"Precision": "{:.4f}", "Recall": "{:.4f}", "F1": "{:.4f}", "Entry accuracy": "{:.4f}"})

by_rule = pd.DataFrame(avm["by_rule"]).rename(
    columns={"rule": "Rule", "n": "Predicted matches", "correct": "Correct", "precision": "Precision"})
rd.table(by_rule, "Table 3.5. Per-rule accuracy of the automatic matcher on the 20 manual papers.",
         {"Precision": "{:.4f}"})

In [ ]:
auto = lbl["auto"]
rules = pd.Series(auto["rules"]).sort_values(ascending=True)
rule_colors = {"none": rd.AXIS, "title_exact": rd.BLUE, "arxiv": rd.AQUA, "doi": rd.ORANGE, "title_fuzzy": rd.YELLOW}

# Parse per-member automatic shares from m3_requirements.json
req_map = {c["check"]: c for c in req["checks"]}
mem_share_dict = ast.literal_eval(req_map["auto_share_per_member"]["actual"])
mem_shares = {m: float(v.rstrip("%")) for m, v in mem_share_dict.items()}

fig = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.16,
    subplot_titles=("Automatic labels by rule (158,622 entries)", "Automatic paper share per member (floor: 10%)"),
)
fig.add_trace(go.Bar(
    y=rules.index.tolist(), x=rules.values.tolist(), orientation="h",
    marker_color=[rule_colors.get(k, rd.BLUE) for k in rules.index],
    text=[f"{v:,} ({100 * v / auto['entries']:.1f}%)" for v in rules.values],
    textposition="outside", showlegend=False,
    hovertemplate="Rule: <b>%{y}</b><br>%{x:,} entries<extra></extra>",
), row=1, col=1)
fig.add_trace(go.Bar(
    x=list(mem_shares.keys()), y=list(mem_shares.values()),
    marker_color=rd.BLUE, text=[f"{v:.1f}%" for v in mem_shares.values()],
    textposition="outside", showlegend=False,
    hovertemplate="%{x}: <b>%{y:.1f}%</b> of non-manual papers<extra></extra>",
), row=1, col=2)
fig.add_hline(y=10.0, line_dash="dash", line_color=rd.ORANGE, line_width=1.5,
              annotation_text="10% requirement", annotation_position="top right", row=1, col=2)
fig.update_layout(title=f"Automatic labelling output ({auto['papers']:,} papers, {auto['matched']:,} matched pairs)")
fig.update_xaxes(title_text="Entries", range=[0, rules.max() * 1.42], row=1, col=1)
fig.update_yaxes(showgrid=False, row=1, col=1)
fig.update_xaxes(showgrid=False, row=1, col=2)
fig.update_yaxes(title_text="Share of non-manual papers (%)", range=[0, 25], row=1, col=2)
rd.show(fig, height=360, margin=dict(l=90, r=25, t=65, b=45))
rd.figure_caption("Figure 3.4. Left: distribution of automatic labels across rules. Right: automatically labelled "
                  "paper share per member (overall 18.8% of the 19,980 non-manual papers, all above the 10% floor).")

t_sp = pd.DataFrame([
    ("Train", 12, auto["papers"] - 8, 12 + auto["papers"] - 8, "all remaining labelled papers"),
    ("Validation", len(sp["valid"]["manual"]), len(sp["valid"]["auto"]),
     len(sp["valid"]["manual"]) + len(sp["valid"]["auto"]),
     "manual: " + ", ".join(sp["valid"]["manual"]) + " | auto: " + ", ".join(sp["valid"]["auto"])),
    ("Test", len(sp["test"]["manual"]), len(sp["test"]["auto"]),
     len(sp["test"]["manual"]) + len(sp["test"]["auto"]),
     "manual: " + ", ".join(sp["test"]["manual"]) + " | auto: " + ", ".join(sp["test"]["auto"])),
], columns=["Partition", "Manual papers", "Auto papers", "Total papers", "Publication IDs (seed 42)"])
rd.table(t_sp, "Table 3.6. Publication-level train / valid / test split (`m3_split.json`).")

## 3.4 Cleaning completeness and feature justification

Before computing features, both sides (`entries` and `refs`) pass through shared cleaning functions. Below we
check field availability after cleaning (`m3_cleaning_report.csv`) and quantify the discriminative power of
each of the 25 features across all 4,821,691 candidate pairs (`m3_feature_report.csv`).

In [ ]:
c_tbl = clean.copy()
pct_cols = ["has_title", "has_authors", "has_year", "has_arxiv_hint", "has_doi", "arxiv_venue", "has_context"]
for col in pct_cols:
    c_tbl[col] = 100 * c_tbl[col]
rd.table(c_tbl, "Table 3.7. Field completeness after cleaning (`m3_cleaning_report.csv`).",
         {c: "{:.2f}%" for c in pct_cols})

c_plot = clean[clean["group"].isin(["bib", "bibitem"])].set_index("group")
fields_p = ["has_title", "has_authors", "has_year", "has_arxiv_hint", "has_doi", "has_context"]
fig = go.Figure()
for grp, color in (("bib", rd.BLUE), ("bibitem", rd.ORANGE)):
    vals = [100 * c_plot.loc[grp, f] for f in fields_p]
    fig.add_trace(go.Bar(
        x=fields_p, y=vals, name=f".{grp}" if grp == "bib" else "\\bibitem",
        marker_color=color, text=[f"{v:.1f}%" for v in vals], textposition="outside",
        hovertemplate=f"{grp} · %{{x}}<br><b>%{{y:.2f}}%</b> present<extra></extra>",
    ))
fig.update_layout(
    barmode="group",
    title="Cleaned field completeness: .bib vs \\bibitem entries",
    legend=dict(orientation="h", yanchor="top", y=-0.22, xanchor="center", x=0.5),
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(title_text="Entries with field (%)", range=[0, 114])
rd.show(fig, height=350, margin=dict(l=55, r=25, t=55, b=75))
rd.figure_caption("Figure 3.5. Field availability in cleaned BibTeX entries by bibliography source.")

In [ ]:
f_sorted = feat.sort_values("auc_oriented", ascending=True).reset_index(drop=True)
groups = list(dict.fromkeys(feat["group"]))
g_color = {g: rd.SERIES[i % len(rd.SERIES)] for i, g in enumerate(groups)}

fig = go.Figure()
# View 0: Oriented AUC
fig.add_trace(go.Bar(
    y=f_sorted["feature"], x=f_sorted["auc_oriented"], orientation="h",
    marker_color=[g_color[g] for g in f_sorted["group"]],
    text=[f"{v:.4f}" for v in f_sorted["auc_oriented"]], textposition="outside",
    customdata=np.stack([f_sorted["group"], f_sorted["direction"], f_sorted["mrr5_skip"], f_sorted["row_constant"]], axis=-1),
    hovertemplate="<b>%{y}</b> (%{customdata[0]})<br>Oriented AUC: <b>%{x:.4f}</b> (%{customdata[1]})<br>"
                  "Single-feature MRR@5 (skip): %{customdata[2]:.4f}<br>Row-constant: %{customdata[3]}<extra></extra>",
    visible=True, showlegend=False,
))
# View 1: Single-feature MRR@5 (skip)
f_mrr = feat.sort_values("mrr5_skip", ascending=True).reset_index(drop=True)
fig.add_trace(go.Bar(
    y=f_mrr["feature"], x=f_mrr["mrr5_skip"], orientation="h",
    marker_color=[g_color[g] for g in f_mrr["group"]],
    text=[f"{v:.4f}" for v in f_mrr["mrr5_skip"]], textposition="outside",
    customdata=np.stack([f_mrr["group"], f_mrr["auc_oriented"], f_mrr["row_constant"]], axis=-1),
    hovertemplate="<b>%{y}</b> (%{customdata[0]})<br>Single-feature MRR@5 (skip): <b>%{x:.4f}</b><br>"
                  "Oriented AUC: %{customdata[1]:.4f}<br>Row-constant: %{customdata[2]}<extra></extra>",
    visible=False, showlegend=False,
))

fig.update_layout(
    title="Feature discriminative power across 4,821,691 candidate pairs (coloured by group)",
    updatemenus=[dict(
        type="buttons", direction="right", active=0, x=1.0, xanchor="right", y=1.10, yanchor="top",
        buttons=[
            dict(label="Oriented AUC", method="update",
                 args=[{"visible": [True, False]}, {"xaxis.title": "Oriented ROC-AUC", "xaxis.range": [0, 1.12]}]),
            dict(label="Single-feature MRR@5 (skip)", method="update",
                 args=[{"visible": [False, True]}, {"xaxis.title": "Single-feature MRR@5 (skip)", "xaxis.range": [0, 1.12]}]),
        ],
    )],
)
fig.update_xaxes(title_text="Oriented ROC-AUC", range=[0, 1.12])
fig.update_yaxes(showgrid=False)
rd.show(fig, height=640, margin=dict(l=150, r=30, t=75, b=45))
rd.figure_caption("Figure 3.6. All 25 engineered features ranked by oriented AUC or single-feature MRR@5 (toggle "
                  "buttons top-right). Row-constant features cannot rank candidates alone but inform NIL detection "
                  "and learned weighting.")

f_disp = feat[["feature", "group", "coverage", "row_constant", "mean_pos", "mean_neg", "direction", "auc_oriented", "mrr5_skip"]].copy()
f_disp["coverage"] = 100 * f_disp["coverage"]
f_disp.columns = ["Feature", "Group", "Coverage", "Row const", "Mean (+)", "Mean (−)", "Direction", "Oriented AUC", "MRR@5 (skip)"]
rd.table(f_disp, "Table 3.8. Per-feature statistics on the 4,821,691 labelled candidate pairs (`m3_feature_report.csv`).",
         {"Coverage": "{:.2f}%", "Mean (+)": "{:.4f}", "Mean (−)": "{:.4f}", "Oriented AUC": "{:.4f}", "MRR@5 (skip)": "{:.4f}"})